# Fase 2C — ML: análisis de la representación (palabras vs. caracteres vs. ambos)

**Nested CV 5×3 agrupada · LR y SVM lineal · F1-Macro**

**Pregunta.** ¿Mejora el ML clásico si, además de palabras, se usan **n-gramas de caracteres**? Es la única mejora con evidencia directa en IroSvA 2019: los sistemas en 2.º lugar (CIMAT; Miranda-Belmonte & López-Monroy, 2019) y 3.º (JZaragoza) usaron n-gramas de caracteres y palabras con SVM (Ortega-Bueno et al., 2019). Los n-gramas de caracteres capturan alargamientos («siiii»), risas («jajaja»), emojis, hashtags y faltas de ortografía, frecuentes en los tuits irónicos. La SVM lineal sobre TF-IDF es además el baseline clásico más fuerte para textos cortos (Wang & Manning, 2012).

**Qué se mantiene igual que en la v1** (`ML_TFIDF_10k_GroupNestedCV_5x3_CORREGIDO`): CSV, grupos (Union-Find), semillas y, por tanto, **los mismos Outer/Inner folds**; preprocesamiento `normal/stem/lemma` elegido en el Inner CV; nueve rasgos lingüísticos en `passthrough`; `class_weight="balanced"`; selección por máximo F1 interno; test bloqueado.

**Qué se compara** (6 configuraciones, definidas antes de correr):

| Representación | Contenido |
|---|---|
| `palabras` | TF-IDF de palabras de la v1 (stopwords NLTK, ≤10k términos, n-gramas (1,1) o (1,2)) |
| `caracteres` | TF-IDF de n-gramas de caracteres `char_wb` de 2 a 5, `sublinear_tf=True`, `min_df=2` |
| `palabras+caracteres` | Ambas concatenadas |

× clasificador **LR** (`liblinear`) y **SVM lineal** (`LinearSVC`). Rejilla de `C`: {0.01, 0.1, 1, 10} (incluye la de la v1) y, cuando hay palabras, n-gramas (1,1)/(1,2).

**Regla de decisión (fijada antes de ver resultados).** Una representación con caracteres se considera mejor que `palabras` solo si, con el mismo clasificador, **sube el F1 medio de la Nested CV en ≥ 0.01 en el promedio de las tres variantes** y la prueba *t* corregida de Nadeau & Bengio (2003) da *p* < 0.05. Si no, la conclusión es que la representación de caracteres no aporta y se mantiene la v1. El test oficial se evalúa una sola vez al final y **no** interviene en la decisión.

Resultados en `../data/resultados_ml_charword_5x3/`.

## 1. Librerías y entorno

In [ ]:
import os
import json
import time
import html
import copy
import warnings
from datetime import datetime
from pathlib import Path

os.environ.setdefault("PYTHONWARNINGS", "ignore")
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import nltk
import matplotlib.pyplot as plt
from scipy import stats
from nltk.corpus import stopwords
from IPython.display import display, HTML

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, classification_report, confusion_matrix,
)

try:
    STOP_WORDS = stopwords.words("spanish")
except LookupError:
    if not nltk.download("stopwords", quiet=True):
        raise RuntimeError("Instala el corpus NLTK de stopwords para reproducir el TF-IDF original.")
    STOP_WORDS = stopwords.words("spanish")

VERSIONES = {"numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
             "sklearn": sklearn.__version__}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))

## 2. Configuración

- `MODO="prueba"`: valida el pipeline en minutos (solo MX, Nested 2×2). Escribe en `…/resultados_ml_charword_5x3/prueba/`.
- `MODO="completo"`: experimento definitivo 5×3.

El test oficial solo se ejecuta en modo completo, con `EJECUTAR_TEST_OFICIAL=True` y `CONFIRMACION_TEST="TEST_OFICIAL_CONGELADO"`.

In [ ]:
DATA_DIR = Path("../data")
RANDOM_STATE = 42
PREPROCESAMIENTOS = {"normal": "", "stem": "_stem", "lemma": "_lemma"}
FEATURE_COLS = ["n_exc", "n_int", "n_may", "n_emo", "n_ris", "n_neg", "n_elo", "n_com", "n_pun"]

REPRESENTACIONES = ["palabras", "caracteres", "palabras+caracteres"]
CLASIFICADORES = ["LR", "SVM"]
CONFIGS = [(r, c) for c in CLASIFICADORES for r in REPRESENTACIONES]

MODO = "completo"
if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER, N_INNER = 2, 2
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER, N_INNER = 5, 3
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

OUT_DIR = DATA_DIR / "resultados_ml_charword_5x3"
V1_DIR = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k"  # solo lectura
if MODO == "prueba":
    OUT_DIR, V1_DIR = OUT_DIR / "prueba", V1_DIR / "prueba"
MODELS_DIR = OUT_DIR / "modelos_finales"
for carpeta in [OUT_DIR, MODELS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

N_JOBS_GRID = -1

# ---------------- Representaciones ----------------
TFIDF_PALABRAS = dict(stop_words=STOP_WORDS, max_features=10000)            # igual que la v1
TFIDF_CARACTERES = dict(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, lowercase=True)

# ---------------- Rejilla ----------------
C_GRID = [0.01, 0.1, 1.0, 10.0]
NGRAMAS_PALABRAS = [(1, 1), (1, 2)]

# ---------------- Regla de decisión (fijada de antemano) ----------------
UMBRAL_MEJORA = 0.01   # ΔF1 medio (promedio de las 3 variantes) frente a `palabras`, mismo clasificador
ALFA = 0.05            # p de la t corregida de Nadeau & Bengio (2003)

EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

print(f"[CONFIG][CharWord] modo={MODO} | variantes={VARIANTES} | Nested={N_OUTER}×{N_INNER}")
print(f"[CONFIG][CharWord] configuraciones={CONFIGS}")
print(f"[CONFIG][CharWord] caracteres={TFIDF_CARACTERES}")
print(f"[CONFIG][CharWord] regla: ΔF1 ≥ {UMBRAL_MEJORA} y p < {ALFA} | test={EJECUTAR_TEST_OFICIAL}")
print(f"[CONFIG][CharWord] OUT_DIR={OUT_DIR.resolve()}")

NOMBRE_EXPERIMENTO = "ML CharWord"
HEARTBEAT_SEGUNDOS = 30
AVISO_SONORO = True


class Monitor:
    """Panel único de progreso + monitor.log, con estado activo / terminado / error.

    Clase común a todos los notebooks de modelos (ML, Hard Voting y Transformers).
    """

    def __init__(self, ruta, total, ya_completados=0, unidad="Fine-tunings completados"):
        self.ruta = Path(ruta)
        self.ruta.mkdir(parents=True, exist_ok=True)
        self.log = self.ruta / "monitor.log"
        self.total = int(total)
        self.unidad = unidad
        self.completados = int(ya_completados)
        self.completados_inicio = int(ya_completados)
        self.inicio = time.monotonic()
        self.ultimo_pulso = 0
        self.etapa = "Inicializando"
        self.estado = "activo"  # activo | terminado | error
        self.handle = display(HTML(f"<b>Preparando {NOMBRE_EXPERIMENTO}...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def registrar(self, msg):
        with self.log.open("a", encoding="utf-8") as f:
            f.write(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n")

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        ahora = time.monotonic()
        if not forzar and ahora - self.ultimo_pulso < HEARTBEAT_SEGUNDOS:
            return
        self.ultimo_pulso = ahora
        minutos = (ahora - self.inicio) / 60
        hechos_sesion = self.completados - self.completados_inicio
        faltan = max(0, self.total - self.completados)
        eta = (
            f"~{minutos / hechos_sesion * faltan:.0f} min"
            if hechos_sesion >= 2
            else "calculando..."
        )
        marca = datetime.now().strftime("%H:%M:%S")
        color, titulo = {
            "activo": ("#777", f"{NOMBRE_EXPERIMENTO} — seguimiento en vivo"),
            "terminado": ("#2e7d32", f"✅ {NOMBRE_EXPERIMENTO} — TERMINADO"),
            "error": ("#c62828", f"❌ {NOMBRE_EXPERIMENTO} — DETENIDO"),
        }[self.estado]
        grosor = 1 if self.estado == "activo" else 3
        tiempo = (
            f"Tiempo: {minutos:.1f} min | ETA orientativo: {eta}"
            if self.estado == "activo"
            else f"Duración total: {minutos:.1f} min | Finalizó a las {marca}"
        )
        panel = (
            f'<div style="font-family:system-ui;padding:12px;border:{grosor}px solid {color};'
            'border-radius:8px;max-width:800px">'
            f'<b style="color:{color}">{titulo}</b><br>'
            f"Etapa: <b>{html.escape(self.etapa)}</b><br>"
            f"{self.unidad}: {self.completados}/{self.total} "
            f"({100*self.completados/max(1,self.total):.1f}%)<br>"
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f"{tiempo}<br>"
            f"Última señal: {marca}<br>Log: {html.escape(str(self.log))}</div>"
        )
        self.handle.update(HTML(panel))

    def iniciar(self, msg):
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def pulso(self):
        if time.monotonic() - self.ultimo_pulso >= HEARTBEAT_SEGUNDOS:
            self.registrar("ACTIVO: " + self.etapa)
            self.mostrar(forzar=True)

    def fit_terminado(self, msg):
        self.completados += 1
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def finalizar(self, msg):
        """Etapa terminada: panel verde al 100 %, mensaje en la celda y aviso sonoro."""
        self.registrar(f"FINALIZADO: {msg} ({self.completados}/{self.total} contados)")
        self.estado = "terminado"
        self.completados = max(self.completados, self.total)
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n✅ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=True)

    def fallar(self, msg):
        """Etapa detenida (error o interrupción): panel rojo, mensaje en la celda y aviso sonoro."""
        self.registrar(f"DETENIDO: {msg}")
        self.estado = "error"
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n❌ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=False)


def avisar_fin(ok=True):
    """Pitido de Windows al terminar (3 veces) o al detenerse (1 vez); en otros sistemas no hace nada."""
    if not AVISO_SONORO:
        return
    try:
        import winsound

        for _ in range(3 if ok else 1):
            winsound.MessageBeep(winsound.MB_ICONASTERISK if ok else winsound.MB_ICONHAND)
            time.sleep(0.5)
    except Exception:
        pass

## 3. Integridad de los CSV y validación por grupos

Celda idéntica a la de la v1: mismos grupos y mismos splits `StratifiedGroupKFold`, por lo que los Outer Folds coinciden con los de la v1 y los Transformers.

In [ ]:
# ============================================================
# 3. CARGA, INTEGRIDAD Y GRUPOS SIN FUGA ENTRE FOLDS
# ============================================================
GRUPOS_TRAIN = {}  # Variante -> np.array; solo contiene información del TRAIN
AUDITORIA_GRUPOS = []
AUDITORIA_CONFLICTOS = []
AUDITORIA_PARTICIONES = []


class UnionFind:
    """Componentes conexos: unión por rango con compresión de caminos."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def cargar_split(variante, prep, split="train"):
    """Carga el CSV preprocesado de una variante, representación y split."""
    if variante not in VARIANTES or prep not in PREPROCESAMIENTOS:
        raise ValueError(f"Variante/preprocessing no reconocido: {variante}/{prep}")
    ruta = DATA_DIR / f"{split}_clean{PREPROCESAMIENTOS[prep]}_{variante}.csv"
    if not Path(ruta).is_file():
        raise FileNotFoundError(f"Falta el CSV: {ruta}")
    # Preservar los IDs originales: no forzarles una conversión numérica.
    return pd.read_csv(ruta, dtype={"ID": "string"}).reset_index(drop=True)


def preparar_xy(df):
    """Separa texto + 9 rasgos (X) y etiqueta (y)."""
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).reset_index(drop=True)
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    return X.reset_index(drop=True), y


def texto_clave(serie):
    """Normalización usada solo para comparar textos al agrupar (no altera el texto del modelo)."""
    # Misma normalización de COMPARACIÓN que la auditoría previa; no modifica el texto del modelo.
    return serie.fillna("").astype(str).str.casefold().str.strip()


def construir_grupos(variante, dfs_por_prep, y):
    """Componentes conexos: agrupa un mismo texto tras normal, stem O lemma.
    No deduplica, no usa las etiquetas para construir grupos y conserva rasgos originales.
    Añade coincidencias de MESSAGE original exacto, si está disponible.
    """
    n = len(y)
    uf = UnionFind(n)

    for prep, df in dfs_por_prep.items():
        primero = {}
        for i, clave in enumerate(texto_clave(df["MESSAGE_CLEAN"])):
            if not clave:
                raise ValueError(
                    f"{variante}/{prep}: texto limpio vacío en fila {i}; revisar antes de agrupar"
                )
            if clave in primero:
                uf.union(primero[clave], i)
            else:
                primero[clave] = i
    # Si dos textos originales son idénticos, deben permanecer en el mismo grupo.
    df_ref = dfs_por_prep["normal"]
    if "MESSAGE" in df_ref:
        primero_raw = {}
        for i, clave in enumerate(texto_clave(df_ref["MESSAGE"])):
            if not clave:
                continue
            if clave in primero_raw:
                uf.union(primero_raw[clave], i)
            else:
                primero_raw[clave] = i

    miembros = {}
    for i in range(n):
        miembros.setdefault(uf.find(i), []).append(i)
    # Id de grupo basado en el menor índice original; determinista entre ambos notebooks.
    ids = {ra: min(idx) for ra, idx in miembros.items()}
    grupos = np.array([ids[uf.find(i)] for i in range(n)], dtype=int)
    duplicados = {ids[ra]: idx for ra, idx in miembros.items() if len(idx) > 1}
    conflictivos = [(g, idx) for g, idx in duplicados.items() if y.iloc[idx].nunique() > 1]
    if conflictivos:
        warnings.warn(
            f"{variante.upper()}: {len(conflictivos)} grupo(s) con etiquetas contradictorias. "
            "Se conservan las etiquetas originales y se mantienen juntos todos sus registros. "
            "Revisar manualmente antes de interpretar resultados.",
            stacklevel=2,
        )
    for g, idx in conflictivos:
        for i in idx:
            AUDITORIA_CONFLICTOS.append(
                {
                    "variante": variante,
                    "grupo": int(g),
                    "fila_0base": int(i),
                    "id_original": str(df_ref.iloc[i]["ID"]) if "ID" in df_ref else "",
                    "etiqueta": int(y.iloc[i]),
                    "texto_original": str(df_ref.iloc[i]["MESSAGE"]) if "MESSAGE" in df_ref else "",
                    "texto_normal": str(df_ref.iloc[i]["MESSAGE_CLEAN"]),
                }
            )
    AUDITORIA_GRUPOS.append(
        {
            "variante": variante,
            "n_train": n,
            "n_grupos": len(miembros),
            "n_grupos_multiples": len(duplicados),
            "n_filas_en_grupos_multiples": sum(map(len, duplicados.values())),
            "tamano_grupo_max": max(map(len, miembros.values())),
            "n_grupos_etiquetas_contradictorias": len(conflictivos),
            "criterio": "union_texto_normal_stem_lemma_y_original_exacto",
        }
    )
    return grupos


def cargar_todos_preps(variante, split="train"):
    """Carga normal/stem/lemma de una variante, verifica que estén alineados y construye los grupos."""
    dfs = {p: cargar_split(variante, p, split=split) for p in PREPROCESAMIENTOS}
    ref = dfs["normal"]
    X_ref, y_ref = preparar_xy(ref)
    datos = {"normal": X_ref}
    for prep, df in dfs.items():
        if prep == "normal":
            continue
        X, y = preparar_xy(df)
        if len(df) != len(ref) or not y.equals(y_ref):
            raise ValueError(f"{variante}/{prep}: diferente número de filas o etiquetas")
        for c in ["ID", "MESSAGE"]:
            if c in ref.columns and c not in df.columns:
                raise ValueError(f"{variante}/{prep}: falta columna de alineación {c}")
            if c in ref.columns and not ref[c].astype(str).equals(df[c].astype(str)):
                raise ValueError(f"{variante}/{prep}: columna {c} desalineada")
        if not X[FEATURE_COLS].equals(X_ref[FEATURE_COLS]):
            raise ValueError(f"{variante}/{prep}: rasgos lingüísticos no alineados")
        datos[prep] = X
    if split == "train" and variante not in GRUPOS_TRAIN:
        GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)
    return datos, y_ref


def grupos_train(variante):
    """Devuelve los grupos del train de una variante (los construye si aún no existen)."""
    if variante not in GRUPOS_TRAIN:
        cargar_todos_preps(variante, split="train")
    return GRUPOS_TRAIN[variante]


def verificar_particion(grupos, a, b, variante, nivel, outer, inner=None):
    """Comprueba que train y validación no compartan grupos y registra la partición."""
    solapamiento = set(grupos[a]) & set(grupos[b])
    if solapamiento:
        raise AssertionError(
            f"FUGA: {variante}/{nivel}/{outer}/{inner} grupos compartidos: {solapamiento}"
        )
    AUDITORIA_PARTICIONES.append(
        {
            "variante": variante,
            "nivel": nivel,
            "outer_fold": outer,
            "inner_fold": inner,
            "n_train": len(a),
            "n_validacion": len(b),
            "n_grupos_compartidos": 0,
        }
    )


def crear_splits_outer(y, grupos, variante="?"):
    """Outer CV agrupada y estratificada."""
    cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "outer", fold)
    return splits


def crear_splits_inner(y, grupos, outer_fold, variante="?"):
    """Inner CV agrupada dentro de un Outer Train (semilla RANDOM_STATE + outer_fold)."""
    cv = StratifiedGroupKFold(
        n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE + outer_fold
    )
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "inner", outer_fold, fold)
    return splits


def crear_splits_final(y, grupos, variante="?"):
    """CV agrupada sobre todo el train para la búsqueda final."""
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "final", 0, fold)
    return splits


# ============================================================
# AUDITORÍA PREVIA — SOLO TRAIN Y FOLDS
# ============================================================
for v in VARIANTES:
    _, etiquetas = cargar_todos_preps(v, split="train")
    gv = grupos_train(v)

    for o, (a, b) in enumerate(crear_splits_outer(etiquetas, gv, v), 1):
        crear_splits_inner(etiquetas.iloc[a].reset_index(drop=True), gv[a], o, v)

    crear_splits_final(etiquetas, gv, v)


df_auditoria_grupos = pd.DataFrame(AUDITORIA_GRUPOS)

df_auditoria_particiones = pd.DataFrame(AUDITORIA_PARTICIONES).drop_duplicates()

df_auditoria_conflictos = pd.DataFrame(AUDITORIA_CONFLICTOS)

print("[AUDIT][ML] Grupos del train")
display(df_auditoria_grupos)

print("\n[AUDIT][ML] Grupos compartidos entre train y validación (todos deben ser 0):")
display(
    df_auditoria_particiones.groupby(["variante", "nivel"], as_index=False)[
        "n_grupos_compartidos"
    ].max()
)

pd.DataFrame(AUDITORIA_GRUPOS).to_csv(OUT_DIR / "auditoria_grupos.csv", index=False)

df_auditoria_particiones.to_csv(OUT_DIR / "auditoria_folds.csv", index=False)

df_auditoria_conflictos.to_csv(OUT_DIR / "auditoria_etiquetas_contradictorias.csv", index=False)

print("\nPRE-FLIGHT DE DATOS: OK")

## 4. Pipelines y búsqueda

En cada Outer Fold, para cada configuración (representación × clasificador), se ejecuta un `GridSearchCV` (Inner CV agrupada, F1-Macro) por cada preprocesamiento; gana el de mayor F1 interno (empate → menor desviación), igual que en la v1. El estimador elegido se reajusta sobre todo el Outer Train y se evalúa en el Outer Test.

Para LR se guarda la probabilidad de ironía; para la SVM, el valor de `decision_function` (sirve para curvas ROC/PR, no es una probabilidad).

In [ ]:
def crear_preprocesador(representacion, ngram_palabras=(1, 1)):
    transformadores = []
    if "palabras" in representacion:
        transformadores.append(("palabras", TfidfVectorizer(ngram_range=ngram_palabras, **TFIDF_PALABRAS), "MESSAGE_CLEAN"))
    if "caracteres" in representacion:
        transformadores.append(("caracteres", TfidfVectorizer(**TFIDF_CARACTERES), "MESSAGE_CLEAN"))
    transformadores.append(("linguisticas", "passthrough", FEATURE_COLS))
    return ColumnTransformer(transformers=transformadores, remainder="drop")


def crear_pipeline(representacion, clasificador):
    if clasificador == "LR":
        clf = LogisticRegression(solver="liblinear", max_iter=10000, random_state=RANDOM_STATE, class_weight="balanced")
    elif clasificador == "SVM":
        clf = LinearSVC(class_weight="balanced", max_iter=20000, random_state=RANDOM_STATE)
    else:
        raise ValueError(clasificador)
    return Pipeline([("features", crear_preprocesador(representacion)), ("clf", clf)])


def rejilla(representacion):
    grid = {"clf__C": C_GRID}
    if "palabras" in representacion:
        grid["features__palabras__ngram_range"] = NGRAMAS_PALABRAS
    return grid


def puntaje(est, X):
    # Probabilidad de ironía (LR) o margen de la SVM.
    clf = est.named_steps["clf"]
    return est.predict_proba(X)[:, 1] if hasattr(clf, "predict_proba") else est.decision_function(X)


def buscar(X_por_prep, y, representacion, clasificador, splits, monitor=None, etiqueta=""):
    mejor = None
    for prep, X in X_por_prep.items():
        search = GridSearchCV(crear_pipeline(representacion, clasificador), rejilla(representacion), cv=splits,
                              scoring="f1_macro", n_jobs=N_JOBS_GRID, refit=True, error_score="raise")
        search.fit(X, y)
        scores = [search.cv_results_[f"split{i}_test_score"][search.best_index_] for i in range(len(splits))]
        cand = {"prep": prep, "inner_f1": float(search.best_score_), "inner_std": float(np.std(scores, ddof=1)),
                "params": search.best_params_, "estimator": search.best_estimator_}
        if (mejor is None or cand["inner_f1"] > mejor["inner_f1"]
                or (np.isclose(cand["inner_f1"], mejor["inner_f1"]) and cand["inner_std"] < mejor["inner_std"])):
            mejor = cand
        if monitor is not None:
            monitor.fit_terminado(f"{etiqueta} | prep={prep} | F1_in={cand['inner_f1']:.4f}")
    return mejor


FILAS_OUTER, OOF_REGISTROS = [], []

## 5. Nested CV (solo train)

**Costo aproximado (modo completo):** 6 configuraciones × 3 preprocesamientos × 5 Outer × 3 variantes = 270 búsquedas pequeñas. Lo más lento es el TF-IDF de caracteres; en CPU suele tomar menos de una hora.

In [ ]:
monitor = Monitor(OUT_DIR, total=len(VARIANTES) * N_OUTER * len(CONFIGS) * len(PREPROCESAMIENTOS),
                  unidad="Búsquedas GridSearch completadas")
try:
    for variante in VARIANTES:
        X_por_prep, y = cargar_todos_preps(variante, split="train")
        grupos = grupos_train(variante)
        for outer_fold, (i_tr, i_te) in enumerate(crear_splits_outer(y, grupos, variante), start=1):
            y_tr, y_te = y.iloc[i_tr].reset_index(drop=True), y.iloc[i_te].reset_index(drop=True)
            X_tr = {p: X.iloc[i_tr].reset_index(drop=True) for p, X in X_por_prep.items()}
            X_te = {p: X.iloc[i_te].reset_index(drop=True) for p, X in X_por_prep.items()}
            splits = crear_splits_inner(y_tr, grupos[i_tr], outer_fold, variante)
            for representacion, clasificador in CONFIGS:
                etiqueta = f"[{variante.upper()}][{clasificador}][{representacion}][Outer {outer_fold}/{N_OUTER}]"
                monitor.iniciar(etiqueta)
                m = buscar(X_tr, y_tr, representacion, clasificador, splits, monitor, etiqueta)
                est, prep = m["estimator"], m["prep"]
                pred_tr, pred_te = est.predict(X_tr[prep]), est.predict(X_te[prep])
                f1_tr = f1_score(y_tr, pred_tr, average="macro")
                f1_te = f1_score(y_te, pred_te, average="macro")
                f1_c = f1_score(y_te, pred_te, labels=[0, 1], average=None, zero_division=0)
                FILAS_OUTER.append({
                    "variante": variante, "clasificador": clasificador, "representacion": representacion,
                    "outer_fold": outer_fold, "best_preprocessing": prep, "inner_best_f1": m["inner_f1"],
                    "inner_best_std": m["inner_std"], "train_f1_macro": f1_tr, "outer_f1_macro": f1_te,
                    "generalization_gap": f1_tr - f1_te, "outer_f1_clase_0": f1_c[0], "outer_f1_clase_1": f1_c[1],
                    "n_train": len(i_tr), "n_test": len(i_te), "best_params": m["params"],
                })
                OOF_REGISTROS.append(pd.DataFrame({
                    "variante": variante, "clasificador": clasificador, "representacion": representacion,
                    "outer_fold": outer_fold, "idx_train_oficial": i_te, "y_true": y_te.to_numpy(),
                    "y_pred": np.asarray(pred_te), "puntaje": puntaje(est, X_te[prep]),
                }))
                print(f"{etiqueta} prep={prep} C={m['params']['clf__C']} | F1_tr={f1_tr:.4f} "
                      f"F1_out={f1_te:.4f} gap={f1_tr - f1_te:.4f}")
except KeyboardInterrupt:
    monitor.fallar("Nested CV interrumpida")
    raise
except Exception as exc:
    monitor.registrar(f"ERROR {type(exc).__name__}: {exc}")
    monitor.fallar(f"Nested CV con error {type(exc).__name__}; consultar monitor.log")
    raise
monitor.finalizar("Nested CV completada")
df_outer = pd.DataFrame(FILAS_OUTER)
df_oof = pd.concat(OOF_REGISTROS, ignore_index=True)

## 6. Resultados y regla de decisión

**Prueba estadística.** Los Outer Folds comparten datos de entrenamiento, así que una *t* pareada simple subestima la varianza y da *p* demasiado pequeños (Dietterich, 1998). Se usa la **t corregida de Nadeau & Bengio (2003)**: la varianza de las diferencias se multiplica por `(1/J + n_test/n_train)`, con `J` = número de folds. Se calcula por variante (J = 5) y para las tres variantes juntas (J = 15, aproximación que trata las variantes como bloques). Se añade Wilcoxon como prueba no paramétrica.

In [ ]:
def t_corregida(d, ratio):
    # t de Nadeau & Bengio (2003) para diferencias pareadas por fold; ratio = n_test / n_train.
    d = np.asarray(d, dtype=float)
    J = len(d)
    var = d.var(ddof=1)
    if J < 2 or var == 0:
        return np.nan, np.nan
    t = d.mean() / np.sqrt((1 / J + ratio) * var)
    return float(t), float(2 * stats.t.sf(abs(t), df=J - 1))


resumen = (df_outer.groupby(["variante", "clasificador", "representacion"], as_index=False)
           .agg(f1_train_mean=("train_f1_macro", "mean"), f1_nested_mean=("outer_f1_macro", "mean"),
                f1_nested_std=("outer_f1_macro", lambda s: s.std(ddof=1)), gap_mean=("generalization_gap", "mean"),
                f1_clase_1_mean=("outer_f1_clase_1", "mean")))
print("RESUMEN NESTED CV")
display(resumen.round(4))
print("\nF1 NESTED (media de las 3 variantes)")
display(resumen.pivot_table(index="clasificador", columns="representacion", values="f1_nested_mean").round(4))

ratio = float((df_outer["n_test"] / df_outer["n_train"]).mean())
filas, decisiones = [], []
for clasificador in CLASIFICADORES:
    base = df_outer[(df_outer.clasificador == clasificador) & (df_outer.representacion == "palabras")]
    for representacion in ["caracteres", "palabras+caracteres"]:
        otro = df_outer[(df_outer.clasificador == clasificador) & (df_outer.representacion == representacion)]
        par = base.merge(otro, on=["variante", "outer_fold"], suffixes=("_base", "_nuevo"))
        par["d_f1"] = par["outer_f1_macro_nuevo"] - par["outer_f1_macro_base"]
        par["d_gap"] = par["generalization_gap_nuevo"] - par["generalization_gap_base"]
        for variante, g in list(par.groupby("variante")) + [("todas", par)]:
            _, p_c = t_corregida(g["d_f1"], ratio)
            p_w = float(stats.wilcoxon(g["d_f1"]).pvalue) if len(g) >= 5 and not np.allclose(g["d_f1"], 0) else np.nan
            filas.append({"clasificador": clasificador, "representacion": representacion, "variante": variante,
                          "n_folds": len(g), "d_f1_media": g["d_f1"].mean(), "p_t_corregida": p_c,
                          "p_wilcoxon": p_w, "d_gap_media": g["d_gap"].mean()})
        d_medio = par.groupby("variante")["d_f1"].mean().mean()
        _, p_todas = t_corregida(par["d_f1"], ratio)
        decisiones.append({"clasificador": clasificador, "representacion": representacion,
                           "d_f1_promedio_variantes": d_medio, "p_t_corregida": p_todas,
                           "mejora": bool(d_medio >= UMBRAL_MEJORA and p_todas < ALFA)})
df_pruebas = pd.DataFrame(filas)
df_decision = pd.DataFrame(decisiones)
print("\nCOMPARACIÓN FRENTE A `palabras` (mismo clasificador, mismos folds)")
display(df_pruebas.round(4))
print(f"\nREGLA DE DECISIÓN (ΔF1 ≥ {UMBRAL_MEJORA} y p < {ALFA})")
display(df_decision.round(4))
if df_decision["mejora"].any():
    print("→ Al menos una representación con caracteres cumple la regla:",
          df_decision.loc[df_decision.mejora, ["clasificador", "representacion"]].values.tolist())
else:
    print("→ Ninguna representación con caracteres cumple la regla: se mantiene la v1 (palabras).")

# Control: `palabras` + LR debería reproducir aproximadamente la LR de la v1 (misma representación y folds).
ruta_v1 = V1_DIR / "outer_folds_ml.csv"
if ruta_v1.is_file():
    v1 = pd.read_csv(ruta_v1).query("modelo == 'LR'")[["variante", "outer_fold", "outer_f1_macro"]]
    ctrl = df_outer.query("clasificador == 'LR' and representacion == 'palabras'").merge(
        v1, on=["variante", "outer_fold"], suffixes=("", "_v1"))
    print("\nCONTROL: LR + palabras frente a LR v1 (diferencias esperadas pequeñas; la rejilla añade C=0.01)")
    display(ctrl.groupby("variante")[["outer_f1_macro", "outer_f1_macro_v1"]].mean().round(4))

### 6.1 Figura

F1-Macro de la Nested CV y brecha por configuración y variante (punto = media de 5 Outer Folds, barra = ± d.e., huecos = cada fold).

In [ ]:
COLORES = {"palabras": "#8c564b", "caracteres": "#4c72b0", "palabras+caracteres": "#dd8452"}
rng = np.random.default_rng(RANDOM_STATE)
fig, axes = plt.subplots(2, len(CLASIFICADORES), figsize=(13, 8), squeeze=False)
for fila, (col, titulo) in enumerate([("outer_f1_macro", "F1-Macro (Outer Test)"),
                                       ("generalization_gap", "Brecha F1 train − validación")]):
    for j, clasificador in enumerate(CLASIFICADORES):
        ax = axes[fila, j]
        for k, rep in enumerate(REPRESENTACIONES):
            for v_i, variante in enumerate(VARIANTES):
                vals = df_outer[(df_outer.clasificador == clasificador) & (df_outer.representacion == rep)
                                & (df_outer.variante == variante)][col].to_numpy()
                x = v_i + (k - 1) * 0.22
                ax.scatter(x + rng.uniform(-0.03, 0.03, len(vals)), vals, s=12, facecolors="none",
                           edgecolors=COLORES[rep], alpha=0.8)
                ax.errorbar(x, vals.mean(), yerr=vals.std(ddof=1) if len(vals) > 1 else 0, fmt="o", ms=7,
                            capsize=3, color=COLORES[rep], label=rep if v_i == 0 else None)
        ax.set_xticks(range(len(VARIANTES)), [v.upper() for v in VARIANTES])
        ax.set_title(f"{clasificador} — {titulo}")
        ax.grid(axis="y", alpha=0.25)
axes[0, 0].legend(fontsize=9)
fig.suptitle("Representación del texto en ML — Nested CV 5×3", fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "charword_nested_f1_brecha.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "charword_nested_f1_brecha.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)

### 6.2 Guardar resultados de desarrollo — sin tocar test

In [ ]:
df_outer.to_csv(OUT_DIR / "outer_folds_charword.csv", index=False)
resumen.to_csv(OUT_DIR / "resumen_nested_charword.csv", index=False)
df_pruebas.to_csv(OUT_DIR / "comparacion_vs_palabras.csv", index=False)
df_decision.to_csv(OUT_DIR / "decision_regla.csv", index=False)
df_oof.to_csv(OUT_DIR / "predicciones_oof_charword.csv", index=False)
with open(OUT_DIR / "config_experimento.json", "w", encoding="utf-8") as fh:
    json.dump({"modo": MODO, "variantes": VARIANTES, "n_outer": N_OUTER, "n_inner": N_INNER,
               "random_state": RANDOM_STATE, "configs": CONFIGS, "tfidf_palabras": {**TFIDF_PALABRAS, "stop_words": "NLTK spanish"},
               "tfidf_caracteres": TFIDF_CARACTERES, "c_grid": C_GRID, "ngramas_palabras": NGRAMAS_PALABRAS,
               "regla": {"umbral": UMBRAL_MEJORA, "alfa": ALFA, "prueba": "t corregida Nadeau-Bengio"},
               "group_audit": df_auditoria_grupos.to_dict(orient="records"), "versions": VERSIONES},
              fh, indent=2, default=str, ensure_ascii=False)
print(f"[EXPORT] Resultados de desarrollo en {OUT_DIR}")

## 7. Búsqueda final y modelos finales — todo TRAIN, test cerrado

Para cada variante y configuración se repite la búsqueda sobre todo el train con la CV agrupada final de 3 folds (igual que la v1) y se guarda el modelo reajustado.

In [ ]:
FINAL, filas = {}, []
monitor_final = Monitor(OUT_DIR, total=len(VARIANTES) * len(CONFIGS) * len(PREPROCESAMIENTOS),
                        unidad="Búsquedas GridSearch completadas")
try:
    for variante in VARIANTES:
        X_por_prep, y = cargar_todos_preps(variante, split="train")
        splits = crear_splits_final(y, grupos_train(variante), variante)
        for representacion, clasificador in CONFIGS:
            etiqueta = f"[{variante.upper()}][{clasificador}][{representacion}][Final]"
            monitor_final.iniciar(etiqueta)
            m = buscar(X_por_prep, y, representacion, clasificador, splits, monitor_final, etiqueta)
            FINAL[(variante, representacion, clasificador)] = m
            filas.append({"variante": variante, "clasificador": clasificador, "representacion": representacion,
                          "preprocessing": m["prep"], "best_f1_cv_final": m["inner_f1"],
                          "cv_std_final": m["inner_std"], "best_params": m["params"]})
            nombre = f"{clasificador.lower()}_{representacion.replace('+', '_')}_{variante}_{m['prep']}.pkl"
            joblib.dump(m["estimator"], MODELS_DIR / nombre)
except KeyboardInterrupt:
    monitor_final.fallar("Búsqueda final interrumpida")
    raise
except Exception as exc:
    monitor_final.registrar(f"ERROR {type(exc).__name__}: {exc}")
    monitor_final.fallar(f"Búsqueda final con error {type(exc).__name__}; consultar monitor.log")
    raise
monitor_final.finalizar("Búsqueda final completada")
df_final = pd.DataFrame(filas).sort_values(["variante", "clasificador", "representacion"])
display(df_final)
df_final.to_csv(OUT_DIR / "seleccion_final_charword.csv", index=False)

## 8. Test oficial — bloqueado

Se evalúan las seis configuraciones una sola vez, para transparencia. La decisión de la sección 6 ya está tomada y no se cambia con estos resultados. Archivos: `resultados_test_charword.csv` y `predicciones_test_charword.csv` (con el puntaje para ROC/PR).

In [ ]:
if not EJECUTAR_TEST_OFICIAL:
    print("TEST OFICIAL BLOQUEADO: la decisión se toma solo con la Nested CV.")
else:
    if MODO != "completo":
        raise ValueError("El test oficial requiere MODO='completo'.")
    if CONFIRMACION_TEST != "TEST_OFICIAL_CONGELADO":
        raise ValueError("Bloqueado: confirma explícitamente la evaluación final con CONFIRMACION_TEST.")
    res, preds = [], []
    ref = resumen.set_index(["variante", "clasificador", "representacion"])
    for (variante, representacion, clasificador), m in FINAL.items():
        df_test = cargar_split(variante, m["prep"], split="test")
        X_test, y_test = preparar_xy(df_test)
        pred = m["estimator"].predict(X_test)
        f1_c = f1_score(y_test, pred, labels=[0, 1], average=None, zero_division=0)
        cv = ref.loc[(variante, clasificador, representacion)]
        res.append({"variante": variante, "clasificador": clasificador, "representacion": representacion,
                    "preprocessing": m["prep"], "outer_f1_mean": cv["f1_nested_mean"], "outer_f1_std": cv["f1_nested_std"],
                    "f1_macro_test": f1_score(y_test, pred, average="macro"), "accuracy_test": accuracy_score(y_test, pred),
                    "precision_macro_test": precision_score(y_test, pred, average="macro", zero_division=0),
                    "recall_macro_test": recall_score(y_test, pred, average="macro", zero_division=0),
                    "f1_clase_0": f1_c[0], "f1_clase_1": f1_c[1]})
        preds.append(pd.DataFrame({"variante": variante, "clasificador": clasificador, "representacion": representacion,
                                   "ID": df_test["ID"].astype(str).to_numpy(), "y_true": y_test.to_numpy(),
                                   "y_pred": pred, "puntaje": puntaje(m["estimator"], X_test)}))
    df_test_res = pd.DataFrame(res).sort_values(["variante", "clasificador", "representacion"])
    display(df_test_res.round(4))
    display(df_test_res.pivot_table(index=["clasificador", "representacion"], columns="variante",
                                    values="f1_macro_test").assign(promedio=lambda d: d.mean(axis=1)).round(4))
    df_test_res.to_csv(OUT_DIR / "resultados_test_charword.csv", index=False)
    pd.concat(preds, ignore_index=True).to_csv(OUT_DIR / "predicciones_test_charword.csv", index=False)
    print("EVALUACIÓN FINAL TERMINADA: no cambiar la decisión de la sección 6 con estos resultados.")

## 9. Resumen metodológico y referencias

1. Mismos datos, grupos, semillas y folds que la v1; solo cambia la representación del texto y se añade la SVM lineal.
2. Seis configuraciones fijadas de antemano; selección de hiperparámetros por máximo F1 interno, como en la v1.
3. Regla de decisión fijada antes de correr (ΔF1 ≥ 0.01 y *p* < 0.05 con la *t* corregida de Nadeau & Bengio).
4. El test se evalúa una sola vez y no interviene en la decisión.

**Referencias (revisadas por pares)**

- Dietterich, T. G. (1998). Approximate statistical tests for comparing supervised classification learning algorithms. *Neural Computation, 10*(7), 1895–1923.
- Miranda-Belmonte, H. U., & López-Monroy, A. P. (2019). Early fusion of traditional and deep features for irony detection in Twitter. *IberLEF 2019, CEUR-WS 2421*.
- Nadeau, C., & Bengio, Y. (2003). Inference for the generalization error. *Machine Learning, 52*(3), 239–281.
- Ortega-Bueno, R., Rangel, F., Hernández Farías, D. I., Rosso, P., Montes-y-Gómez, M., & Medina Pagola, J. E. (2019). Overview of the task on irony detection in Spanish variants. *IberLEF 2019, CEUR-WS 2421*.
- Wang, S., & Manning, C. D. (2012). Baselines and bigrams: Simple, good sentiment and topic classification. *Proceedings of ACL*, 90–94.